In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import ScalarFormatter
import numpy as np

%matplotlib inline

mpl.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 8,
    'lines.linewidth': 1.5,
    'lines.markersize': 5,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linewidth': 0.5,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

SINGLE_COL = 3.25

WIDTH_COLORS = {
    256:  '#4477AA',
    512:  '#EE6677',
    1024: '#228833',
    2048: '#CCBB44',
    2560: '#AA3377',
}
WIDTH_MARKERS = {
    256: 'o', 512: 's', 1024: '^', 2048: 'D', 2560: 'v',
}

In [ ]:
# Configuration
WIDTHS      = [256, 512, 1024, 2048, 2560]
BATCH_SIZES = [32, 64, 128, 256, 512, 1024, 2048]
LEARNING_RATES = [0.015625, 0.0078125, 0.00390625, 0.001953125, 0.0009765625, 0.00048828125]
SEED        = 111

X_MIN, X_MAX = 2.6, 5.0

LOSS_VALUES = {
    256:  [4.7, 4.5, 4.2, 4.0],
    512:  [4.7, 4.5, 4.2, 4.0, 3.8],
    1024: [4.7, 4.5, 4.2, 4.0, 3.8, 3.6, 3.4],
    2048: [4.7, 4.5, 4.2, 4.0, 3.8, 3.6, 3.4, 3.2, 3.0],
    2560: [4.7, 4.5, 4.2, 4.0, 3.8, 3.6, 3.4, 3.2, 3.0],
}

BASE_SMOOTHING_WINDOW = 64   # EWM span base (CBS)
GNS_SMOOTHING_WINDOW  = 512  # centered rolling (GNS)
CBS_SPEEDUP_THRESHOLD = 1.0 / 1.2

GNS_CONFIGS = {
    256:  [{'bs': 512,  'lr': 0.0078125}],
    512:  [{'bs': 512,  'lr': 0.0078125}],
    1024: [{'bs': 1024, 'lr': 0.0078125}],
    2048: [{'bs': 1024, 'lr': 0.0078125}],
    2560: [{'bs': 1024, 'lr': 0.0078125}],
}

In [ ]:
# Load parquet
df = pd.read_parquet('nlp.parquet')
print(df.shape)
df.head(2)

In [ ]:
# Helper: get a subset for one run
def get_run(width, bs, lr, seed=SEED):
    return df[
        (df['width']      == width) &
        (df['batch_size'] == bs) &
        (df['peak_lr']    == lr) &
        (df['seed']       == seed)
    ]

_cbs_cache, _gns_cache = {}, {}

def load_for_cbs(width, bs, lr):
    key = (width, bs, lr)
    if key in _cbs_cache:
        return _cbs_cache[key]

    subset = get_run(width, bs, lr)
    subset = subset[subset['train/loss'].notna()].copy()

    if len(subset) == 0:
        _cbs_cache[key] = None
        return None

    span = max(5, int(BASE_SMOOTHING_WINDOW / np.sqrt(bs)))
    subset['loss_smoothed'] = subset['train/loss'].ewm(span=span, adjust=False).mean()
    subset['samples_seen'] = subset['iteration'] * bs
    _cbs_cache[key] = subset
    return subset


def load_for_gns(width, bs, lr):
    key = (width, bs, lr)
    if key in _gns_cache:
        return _gns_cache[key]

    subset = get_run(width, bs, lr)
    subset = subset[subset['gns'].notna()].copy()

    if len(subset) == 0:
        _gns_cache[key] = None
        return None

    subset['loss_smoothed'] = subset['train/loss'].rolling(GNS_SMOOTHING_WINDOW, center=True).mean()
    subset['gns_smoothed']  = subset['gns'].rolling(GNS_SMOOTHING_WINDOW, center=True).mean()
    subset = subset.dropna(subset=['loss_smoothed', 'gns_smoothed'])
    _gns_cache[key] = subset
    return subset

In [ ]:
# Compute CBS for all widths
def find_critical_batch_size(target_loss, width):
    results = []
    for bs in BATCH_SIZES:
        for lr in LEARNING_RATES:
            run = load_for_cbs(width, bs, lr)
            if run is None:
                continue
            mask = run['loss_smoothed'] <= target_loss
            if mask.any():
                samples = run.loc[mask.idxmax(), 'samples_seen']
                results.append({'batch_size': bs, 'lr': lr, 'samples': samples})

    if not results:
        return np.nan

    res = pd.DataFrame(results)
    best = []
    for bs in BATCH_SIZES:
        sub = res[res['batch_size'] == bs]
        if len(sub):
            best.append(sub.loc[sub['samples'].idxmin()])

    if not best:
        return np.nan

    best_df = pd.DataFrame(best).sort_values('batch_size').reset_index(drop=True)
    min_samples = best_df.iloc[0]['samples']
    best_df['speedup'] = min_samples / best_df['samples']

    for i in range(1, len(best_df)):
        if best_df.iloc[i]['speedup'] < CBS_SPEEDUP_THRESHOLD:
            bs1, bs2 = best_df.iloc[i-1]['batch_size'], best_df.iloc[i]['batch_size']
            s1,  s2  = best_df.iloc[i-1]['speedup'],    best_df.iloc[i]['speedup']
            t = (CBS_SPEEDUP_THRESHOLD - s1) / (s2 - s1)
            return np.exp(np.log(bs1) + (np.log(bs2) - np.log(bs1)) * t)

    return best_df.iloc[-1]['batch_size']


cbs_data = {}
for width in WIDTHS:
    print(f'Width {width}...', end=' ')
    cbs_data[width] = [find_critical_batch_size(loss, width) for loss in LOSS_VALUES[width]]
    print(cbs_data[width])

In [ ]:
# Load GNS data
gns_data = {}
for width, configs in GNS_CONFIGS.items():
    gns_data[width] = []
    for cfg in configs:
        run = load_for_gns(width, cfg['bs'], cfg['lr'])
        if run is not None:
            gns_data[width].append(run)
            print(f'Width {width}: loaded GNS bs={cfg["bs"]} lr={cfg["lr"]}')

In [ ]:
# Create joint CBS + GNS plot
fig, ax = plt.subplots(1, 1, figsize=(SINGLE_COL, SINGLE_COL * 0.9))

# GNS (solid lines)
for width in WIDTHS:
    for run in gns_data.get(width, []):
        mask = (run['loss_smoothed'] >= X_MIN) & (run['loss_smoothed'] <= X_MAX)
        r = run[mask]
        if len(r):
            ax.plot(r['loss_smoothed'], r['gns_smoothed'] * 1024,
                    color=WIDTH_COLORS[width], label=f'Width {width}', zorder=2)

# CBS (dashed lines with markers)
for width in WIDTHS:
    pairs = [(l, c * 1024) for l, c in zip(LOSS_VALUES[width], cbs_data[width]) if not np.isnan(c)]
    if pairs:
        losses, vals = zip(*pairs)
        ax.plot(losses, vals,
                marker=WIDTH_MARKERS[width], color=WIDTH_COLORS[width],
                linestyle='--', alpha=0.5,
                markeredgecolor='white', markeredgewidth=0.5, zorder=3)

ax.set_xlabel('Training Loss')
ax.set_ylabel('GNS / CBS (Tokens)')
ax.set_title('GPT / FineWeb', fontweight='medium')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(X_MIN, X_MAX)
ax.set_ylim(10 * 1024, 3000 * 1024)
ax.invert_xaxis()
ax.set_xticks([3, 4, 5])
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.xaxis.get_major_formatter().set_scientific(False)
ax.legend(loc='lower right', ncol=1, columnspacing=0.8,
           handlelength=1, handletextpad=0.4, fontsize=8)

fig.tight_layout()
plt.show()

for fmt in ['pdf', 'png']:
    fig.savefig(f'cbs_gns_joint_nlp.{fmt}', format=fmt,
                pad_inches=0.02, dpi=300 if fmt == 'png' else None)
    print(f'Saved cbs_gns_joint_nlp.{fmt}')